# Лекција 11 - Протокол контекста модела (MCP)

The **Протокол контекста модела (MCP)** је отворени стандард који омогућава агентима да динамички откривају и користе алате, ресурсе и изворе података у време извршавања. Уместо да се алати статички уграђују у агента, MCP омогућава агентима да се повежу са спољним серверима који по потреби откривају могућности.

У овој лекцији ћете научити:
- Шта је MCP и зашто је важан за агентске системе
- Како функционише MCP клијент-сервер архитектура
- Како изградити агенте који користе откривање алата по MCP моделу


## Подешавање

**Преуслови:**
- Пројекат Azure AI Foundry са распоређеним моделом
- Покрените `az login` за аутентификацију


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated
from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Шта је Протокол контекста модела (MCP)?

MCP дефинише стандардни начин за AI агенте да откривају и интерагују са спољним алатима и изворима података:

- **MCP сервер**: Чини доступним алате, ресурсе и упутства преко стандардног протокола
- **MCP клијент**: Извршно окружење агента које се повезује на сервере и открива расположиве могућности
- **Динамичко откривање**: Агенти не морају да имају хардкодиране алате — они откривају шта је доступно током извршавања

Ово је моћно за изградњу проширивих система агената где се нове могућности могу додати без модификовања кода агента.


## Како MCP функционише

```
┌─────────────┐     discover      ┌─────────────────┐
│  MCP Client  │ ──────────────► │   MCP Server     │
│  (Agent)     │                  │  (Tool Provider) │
│              │ ◄────────────── │                   │
│              │   tool results   │  • list_tools()  │
│              │                  │  • call_tool()   │
└─────────────┘                  │  • resources     │
                                  └─────────────────┘
```

1. Агент (MCP клијент) се повезује са MCP сервером
2. Сервер одговара са списком доступних алата и њиховим шемама
3. Агент тада може позвати било који откривени алат током свог расуђивања
4. Резултати се враћају кроз исти протокол


## Симулирање откривања MCP алата

Пошто прави MCP сервер захтева покренут серверски процес, показаћемо образац користећи `@tool` функције које симулирају шта би сервис за смештај повезан са MCP-ом обезбеђивао.

У продукцији, ови алати би били динамички откривени са MCP сервера уместо да буду дефинисани локално.


In [ ]:
@tool(approval_mode="never_require")
def search_accommodations(
    location: Annotated[str, "The city to search for accommodations"],
    check_in: Annotated[str, "Check-in date (YYYY-MM-DD)"],
    check_out: Annotated[str, "Check-out date (YYYY-MM-DD)"],
    guests: Annotated[int, "Number of guests"] = 2
) -> str:
    """Search for accommodations (simulating an MCP-connected Airbnb tool).
    In production, this would be discovered via MCP from an accommodation service."""
    listings = {
        "Tokyo": [
            {"name": "Shinjuku Modern Apartment", "price": 120, "rating": 4.8},
            {"name": "Traditional Ryokan in Asakusa", "price": 200, "rating": 4.9},
            {"name": "Shibuya Studio", "price": 85, "rating": 4.5},
        ],
        "Paris": [
            {"name": "Le Marais Charming Flat", "price": 150, "rating": 4.7},
            {"name": "Montmartre Artist Loft", "price": 110, "rating": 4.6},
        ],
        "Barcelona": [
            {"name": "Gothic Quarter Penthouse", "price": 130, "rating": 4.8},
            {"name": "Barceloneta Beach Flat", "price": 95, "rating": 4.4},
        ],
    }
    results = listings.get(location, [])
    if not results:
        return f"No accommodations found in {location}"
    output = f"Accommodations in {location} ({check_in} to {check_out}, {guests} guests):\n"
    for listing in results:
        output += f"  - {listing['name']}: ${listing['price']}/night (★{listing['rating']})\n"
    return output


@tool(approval_mode="never_require")
def get_local_experiences(
    location: Annotated[str, "The city to find experiences in"],
    interest: Annotated[str, "Type of experience (food, culture, adventure, etc.)"] = "all"
) -> str:
    """Get local experiences and activities (simulating an MCP-connected tourism tool)."""
    experiences = {
        "Tokyo": {
            "food": ["Tsukiji Market Tour ($45)", "Ramen Making Class ($60)", "Sake Tasting ($35)"],
            "culture": ["Tea Ceremony ($50)", "Samurai Museum ($15)", "Sumo Tournament ($80)"],
            "adventure": ["Mt. Fuji Day Trip ($120)", "Go-kart City Tour ($80)"],
        },
        "Paris": {
            "food": ["Wine & Cheese Tasting ($55)", "Cooking Class ($90)", "Market Tour ($40)"],
            "culture": ["Louvre Guided Tour ($35)", "Montmartre Art Walk ($25)"],
        },
    }
    city_exp = experiences.get(location, {})
    if not city_exp:
        return f"No experiences found in {location}"
    if interest != "all" and interest in city_exp:
        items = city_exp[interest]
        return f"{interest.title()} experiences in {location}:\n" + "\n".join(f"  - {e}" for e in items)
    output = f"All experiences in {location}:\n"
    for cat, items in city_exp.items():
        output += f"\n  {cat.title()}:\n"
        for item in items:
            output += f"    - {item}\n"
    return output

## Изградња агента помоћу алата у MCP стилу


In [ ]:
agent = await provider.create_agent(
    tools=[search_accommodations, get_local_experiences],
    name="AccommodationAgent",
    instructions="""You are an accommodation and travel experiences specialist powered by MCP-connected services.

Help travelers find the perfect place to stay and things to do. When searching:
1. Use the search_accommodations tool to find listings
2. Use the get_local_experiences tool to suggest activities
3. Compare options and make personalized recommendations
4. Consider the traveler's budget, interests, and travel style""",
)

response = await agent.run(
    "I'm visiting Tokyo for 5 nights in April with my partner. We love traditional Japanese culture and food. "
    "Find us a place to stay and suggest some experiences.",
    )
print(response)

## MCP у продукцији

У продукционом окружењу, MCP омогућава моћне обрасце:

- **Dynamic tool discovery**: Агенти се повезују на MCP сервере и откривају алате за време извршавања
- **Decoupled architecture**: Провајдери алата могу да ажурирају своје компоненте независно од агента
- **Cross-organization sharing**: Тимови могу да изложе могућности преко MCP сервера које било који агент може користити
- **Microsoft Agent Framework support**: MAF има уграђену подршку за MCP клијента преко интеграције `mcp`

Да бисте користили прави MCP сервер са MAF-ом, повезали бисте се преко `hosted_mcp_tool()` или MCP клијент интеграције.

**Сазнајте више:**
- [MCP Specification](https://modelcontextprotocol.io/)
- [Microsoft Agent Framework MCP Support](https://github.com/microsoft/agent-framework/tree/main/python/samples/02-agents/mcp)


## Резиме

У овој лекцији сте научили:
- **MCP** је отворени стандард за динамичко откривање алата између агената и пружалаца алата
- **архитектура клијент-сервер** омогућава агентима да открију могућности током извршавања
- MCP омогућава **прошириве, раздвојене агентске системе** где се алати могу додати без измена кода
- Microsoft Agent Framework пружа **уграђену подршку за MCP** за употребу у продукцији


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Изјава о одрицању одговорности**:
Овај документ је преведен помоћу услуге за превођење засноване на вештачкој интелигенцији [Co-op Translator](https://github.com/Azure/co-op-translator). Иако тежимо тачности, имајте у виду да аутоматски преводи могу садржати грешке или нетачности. Изворни документ на његовом оригиналном језику треба сматрати меродавним извором. За критичне информације препоручује се професионалан људски превод. Не сносимо одговорност за било каква непоразумевања или погрешна тумачења која проистичу из коришћења овог превода.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
